# 03 意思決定戦略

| 戦略 | 選び方 |
| --- | --- |
| **Fixed** | 最初の分担をずっと続ける |
| **Greedy** | その年の**平均満足度**が最大の分担を、候補から全探索で選ぶ |
| **Fairness-aware** | **平均満足度 − λ × 満足度の差** が最大の分担を選ぶ（公平性を考慮した交渉） |
| **Adaptive** | 一定期間ごと（とストレスイベントの年）に見直し、満足度が下がっていたら今の分担の**近く**だけを調べて少しずつ再調整 |

すべての戦略で**同じ候補集合・同じ制約（実行可能性）・同じ満足度関数**を使います。違うのは「選び方」だけです（RQ4）。
計算：`decision_strategies.py`

> **教育用の架空データ・架空モデル**です。Partner A・B は実在の人物ではなく、性別や役割を意味しません。満足度関数や関係解消確率の係数は架空の値で、実際の離婚率や特定の関係の将来を予測するものではありません。

## 準備：実験環境（最初に必ず実行）

F4 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（F4 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | F4 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（F4 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                 # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "f4-partnership-simulation"     # リポジトリの中の F4 フォルダ（ルートが F4 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_f4_root(p):
    return (Path(p) / "household_model.py").exists() and (Path(p) / "relationship_dynamics.py").exists()


def find_f4_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または F4 のルート
        if is_f4_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_f4_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("F4 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/f4_upload")
    for hit in glob.glob("/content/f4_upload/**/household_model.py", recursive=True):
        if is_f4_root(Path(hit).parent):
            return Path(hit).parent
    return None


F4_ROOT = find_f4_root()
if F4_ROOT is None:
    raise RuntimeError("F4 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(F4_ROOT)
if str(F4_ROOT) not in sys.path:
    sys.path.insert(0, str(F4_ROOT))
if not (F4_ROOT / "textbook" / "max.py").exists():
    print("⚠️ textbook/ がありません。decision_strategies.py などの import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import partners as pt
import household_model as hm
import decision_strategies as ds
import relationship_dynamics as rd
import evaluation as ev
import partnership_charts as pc

pc.setup_style()                       # 文字を大きくするなら pc.setup_style(1.3)
scenarios = pt.load_scenarios()        # シナリオ A〜D（すべて架空）
params = rd.load_parameters()          # モデルのパラメータ（すべて架空）
events = pt.load_stress_events()       # ストレスイベントの組
SC = pt.scenario_by_id(scenarios, "A") # 基準にする初期条件（Scenario A）
P, HH = SC["partners"], SC["household"]
COND0 = dict(HH, stress=params["base_stress"])   # ストレスイベントのない年の生活条件
CANDS = ds.candidate_grid(params["share_step"])  # 分担の候補（全戦略で共通）
SEEDS = range(5)                       # 乱数シード
REPLICATES = 200                       # シードごとの反復回数
YEARS = params["simulation_years"]

SAVE_FIGURES = False                   # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", pc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    data = [{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame(data))
    except ImportError:
        print(ev.format_table(rows, columns))


print("F4_ROOT:", F4_ROOT)
print("シナリオ:", [s["label"] for s in scenarios])
print("モデルのパラメータ（架空）:", {k: params[k] for k in ("alpha", "baseline_hazard", "beta", "gamma",
                                                       "base_stress", "fairness_lambda", "share_step", "simulation_years")})

## 1. 候補集合と全探索

- 候補：$x, y \in \{0.0, 0.1, \dots, 1.0\}$ → $11 \times 11 = 121$ 通り（`share_step` で変更可）
- 最大値を教科書の `max_of`（第2章）で求め、その位置を `seq_search`（第3章）で探す → 同点なら**候補の並び順で最初**（決定的）
- 計算量：家事 $K$ 通り × 生活費 $L$ 通り → 1 年あたり $O(KL)$

In [ ]:
feasible = ds.feasible_candidates(CANDS, P, COND0)
print(f"候補 {len(CANDS)} 通り、うち実行可能 {len(feasible)} 通り")
rows = []
for s in ds.STRATEGIES:
    x, y = ds.decide(s, hm.initial_allocation(SC), P, COND0, params, CANDS, review=True)
    a, b = hm.utilities(x, y, P, COND0, params)
    rows.append({"name": ds.LABELS[s], "x": x, "y": y, "A": a, "B": b, "mean": (a + b) / 2, "diff": abs(a - b),
                 "obj": ds.objective(a, b, "fairness", params["fairness_lambda"])})
table(rows, [("name", "strategy (year 1)", "{}"), ("x", "x", "{:.1f}"), ("y", "y", "{:.1f}"), ("A", "U_A", "{:.3f}"),
             ("B", "U_B", "{:.3f}"), ("mean", "mean", "{:.3f}"), ("diff", "diff", "{:.3f}"),
             ("obj", "mean - λ·diff", "{:.3f}")])
print("Fixed は Scenario A の最初の分担（50:50 / 50:50）、Adaptive は 1 年目に見直した場合")

## 2. 20 年間に選ばれた分担

Scenario A の初期条件で、ストレスイベント（Year 5：A の所得 −20%、Year 8：家事の総時間 +10 時間/週）を入れます。
ここでは**関係が続いた場合**の軌跡を見ます（解消の判定はしない）。

In [ ]:
#@title 選ばれた家事分担率
EVENTS = events["combined"]
EV_YEARS = rd.event_years(EVENTS)
trajs = {s: rd.trajectory(SC, s, params, EVENTS) for s in ds.STRATEGIES}
show(pc.plot_strategy_paths(trajs, "housework_share", "x = Partner A's housework share",
                            "Housework share chosen by each strategy (model)", EV_YEARS), "03_paths_housework")

In [ ]:
#@title 選ばれた生活費の負担率
show(pc.plot_strategy_paths(trajs, "financial_share", "y = Partner A's expense share",
                            "Expense share chosen by each strategy (model)", EV_YEARS), "03_paths_expenses")

## 3. 戦略別の満足度と公平性

20 年間（関係が続いた場合）の平均です。

In [ ]:
#@title 戦略別の満足度
res = ev.compare_strategies(SC, params, EVENTS, ds.STRATEGIES, REPLICATES, SEEDS)
show(pc.plot_strategy_metrics(res, [("first_year_mean_utility", "mean sat. (year 1)"), ("mean_satisfaction", "mean sat. (20y avg)"),
                                    ("min_satisfaction", "min sat. (20y avg)"), ("satisfaction_diff", "difference (20y avg)")],
                              "Satisfaction by strategy (model)"), "03_strategy_satisfaction")

In [ ]:
#@title 戦略別の公平性
show(pc.plot_strategy_metrics(res, [("fairness", "fairness index (workload)")], "Fairness by strategy (model)"),
     "03_strategy_fairness")
table([{"name": ds.LABELS[s], **res[s]} for s in ds.STRATEGIES],
      [("name", "strategy", "{}"), ("mean_satisfaction", "mean", "{:.3f}"), ("min_satisfaction", "min", "{:.3f}"),
       ("satisfaction_diff", "diff", "{:.3f}"), ("fairness", "fairness", "{:.3f}"), ("infeasible_years", "infeasible years", "{}")])

## 4. 学生向け TODO：公平性の重み λ を変える

λ = 0 なら Greedy と同じ、λ = 0.5 なら「最低満足度の最大化」と同じ値になります。

In [ ]:
# TODO 1：λ の値を変えて実行しよう
for lam in [0.0, 0.25, 0.5, 1.0, 2.0]:
    x, y = ds.fairness_allocation(P, COND0, params, CANDS, lam=lam)
    a, b = hm.utilities(x, y, P, COND0, params)
    print(f"λ = {lam:4}: (x, y) = ({x}, {y})  U_A {a:.3f}  U_B {b:.3f}  mean {(a + b) / 2:.3f}  diff {abs(a - b):.3f}")

In [ ]:
# TODO 2：目的関数を変える（例：最低満足度を最大化する）
best = ds.best_candidate(ds.feasible_candidates(CANDS, P, COND0), P, COND0, params, kind="min")
print("最低満足度が最大の分担:", best, [round(u, 3) for u in hm.utilities(*best, P, COND0, params)])

# TODO 3：Adaptive の見直し間隔・しきい値を変えて、戦略を比べる
p2 = rd.with_changes(params, review_interval=1, review_threshold=0.7)
r2 = ev.compare_strategies(SC, p2, EVENTS, ["adaptive"], REPLICATES, SEEDS)["adaptive"]
print(f"Adaptive（毎年見直し）: 平均満足度 {r2['mean_satisfaction']:.3f}  最低 {r2['min_satisfaction']:.3f}")

## 考察（チームで記入）

- **TODO**：Greedy と Fairness-aware が選んだ分担はどう違うか。その違いは誰の満足度に影響したか
- **TODO**：Adaptive は少しずつしか分担を変えない。それは良い点か、悪い点か
- **TODO**：ストレスイベントの後、各戦略は分担をどう変えたか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第6章 単純挿入ソート）
- F4 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界と倫理的配慮）
- データ：`data/sample_partnerships.json`（この教材のための**架空のデータ**）